# 05 · A linked three-statement model

A compact model linking a P&L, a cash-flow build, and a balance sheet,
with grouped sections, subtotals, and a balance check flag.



This showcases `group`, the highlight/total formats, the `BOOLEAN`
format, and a derived row via row arithmetic.

In [ ]:
from dataclasses import dataclass

from finmodel import Model, row, PredefinedFormats as F, PredefinedStyles as S



@dataclass

class Inputs:

    revenue: float

    growth: float

    gross_margin: float

    opex_ratio: float

    tax_rate: float

    capex_ratio: float

    dep_rate: float

    opening_cash: float

    opening_ppe: float

    opening_equity: float

## The model

Sections are organised with `group=`. Subtotals/totals use the
highlighted formats; the balance check uses `BOOLEAN`.

In [ ]:
class ThreeStatement(Model[Inputs]):

    # --- Income statement ---

    @row(group="Income statement", format=F.USD)

    def revenue(self, t):

        return self.inputs.revenue if t == 0 else self.revenue(t-1) * (1 + self.inputs.growth)



    @row(group="Income statement", format=F.USD)

    def gross_profit(self, t):

        return self.revenue(t) * self.inputs.gross_margin



    @row(group="Income statement", format=F.USD)

    def opex(self, t):

        return self.revenue(t) * self.inputs.opex_ratio



    @row(group="Income statement", format=F.USD)

    def depreciation(self, t):

        return self.ppe(t-1) * self.inputs.dep_rate if t > 0 else self.inputs.opening_ppe * self.inputs.dep_rate



    @row(group="Income statement", format=F.SUBTOTAL)

    def ebit(self, t):

        return self.gross_profit(t) - self.opex(t) - self.depreciation(t)



    @row(group="Income statement", format=F.USD)

    def tax(self, t):

        return max(self.ebit(t), 0) * self.inputs.tax_rate



    @row(group="Income statement", format=F.TOTAL)

    def net_income(self, t):

        return self.ebit(t) - self.tax(t)



    # --- Cash flow ---

    @row(group="Cash flow", format=F.USD)

    def capex(self, t):

        return self.revenue(t) * self.inputs.capex_ratio



    @row(group="Cash flow", format=F.TOTAL)

    def free_cash_flow(self, t):

        # NI + non-cash depreciation - capex

        return self.net_income(t) + self.depreciation(t) - self.capex(t)



    # --- Balance sheet ---

    @row(group="Balance sheet", format=F.USD)

    def cash(self, t):

        prev = self.inputs.opening_cash if t == 0 else self.cash(t-1)

        return prev + self.free_cash_flow(t)



    @row(group="Balance sheet", format=F.USD)

    def ppe(self, t):

        prev = self.inputs.opening_ppe if t == 0 else self.ppe(t-1)

        return prev + self.capex(t) - self.depreciation(t)



    @row(group="Balance sheet", format=F.SUBTOTAL)

    def total_assets(self, t):

        return self.cash(t) + self.ppe(t)



    @row(group="Balance sheet", format=F.SUBTOTAL)

    def equity(self, t):

        prev = self.inputs.opening_equity if t == 0 else self.equity(t-1)

        return prev + self.net_income(t)



    @row(group="Checks", format=F.BOOLEAN)

    def balances(self, t):

        # Assets should equal equity in this all-equity model.

        return abs(self.total_assets(t) - self.equity(t)) < 1e-6

In [ ]:
inputs = Inputs(

    revenue=1_000, growth=0.10, gross_margin=0.60, opex_ratio=0.25,

    tax_rate=0.25, capex_ratio=0.08, dep_rate=0.15,

    opening_cash=200, opening_ppe=500, opening_equity=700,

)

model = ThreeStatement(periods=5, inputs=inputs, style=S.CLASSIC_LIGHT)

model.calculate()

model.show()

The `Checks → Balances` row renders green `TRUE` when the balance sheet
ties out in every period. Try `style=S.SLATE_DARK` or `S.PRINT` to see
the same model in other themes, or `model.to_html("three_statement.html")`
to export it.